In [ ]:
# Improved experiment: pretrained Kinetics video features + lightweight style/motion features.
# This notebook intentionally stops at a validated submission file; no second Kaggle submission is made here.
import os, glob, warnings, json
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
from joblib import dump, load
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
import torch
import torch.nn as nn
from torchvision.models.video import r3d_18, R3D_18_Weights

warnings.filterwarnings('ignore')
SEED=42
np.random.seed(SEED)
torch.manual_seed(SEED)
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device, 'torch:', torch.__version__)

def find_root():
    for root,_,files in os.walk('/kaggle/input'):
        if 'train_labels.csv' in files and 'sample_submission.csv' in files:
            return root
    raise FileNotFoundError('competition data not found')
COMP=find_root(); TRAIN=os.path.join(COMP,'train'); TEST=os.path.join(COMP,'test')
labels=pd.read_csv(os.path.join(COMP,'train_labels.csv'))
sample=pd.read_csv(os.path.join(COMP,'sample_submission.csv'))
train_ids=labels.clip_id.astype(str).tolist(); test_ids=sample.clip_id.astype(str).tolist()
print('data:', COMP, labels.shape, sample.shape)

# General-purpose Kinetics-400 weights are permitted by the rules; no crash-specific data is used.
weights=R3D_18_Weights.DEFAULT
video_model=r3d_18(weights=weights)
video_model.fc=nn.Identity()
video_model.eval().to(device)
mean=torch.tensor([0.43216,0.394666,0.37645],device=device).view(1,3,1,1,1)
std=torch.tensor([0.22803,0.22145,0.216989],device=device).view(1,3,1,1,1)

def read_clip(folder, nframes=16):
    paths=sorted(glob.glob(os.path.join(folder,'frame_*.jpg')))
    idx=np.linspace(0,len(paths)-1,nframes).round().astype(int)
    arr=[]
    for i in idx:
        im=cv2.imread(paths[i],cv2.IMREAD_COLOR)
        if im is None: im=np.zeros((224,384,3),np.uint8)
        im=cv2.cvtColor(im,cv2.COLOR_BGR2RGB)
        im=cv2.resize(im,(112,112),interpolation=cv2.INTER_AREA)
        arr.append(im)
    x=torch.from_numpy(np.stack(arr).astype(np.float32)/255.0).permute(0,3,1,2).permute(1,0,2,3)
    return x

def deep_features(ids, root, cache_name):
    cache='/kaggle/working/'+cache_name
    if os.path.exists(cache):
        obj=load(cache)
        if obj.get('ids')==list(ids):
            print('loaded',cache); return obj['X']
    out=[]
    with torch.inference_mode():
        for s in range(0,len(ids),8):
            batch=torch.stack([read_clip(os.path.join(root,cid)) for cid in ids[s:s+8]]).to(device)
            batch=(batch-mean)/std
            z=video_model(batch).detach().float().cpu().numpy()
            out.append(z)
            if (s+len(batch))%160==0 or s+len(batch)==len(ids): print('deep features',s+len(batch),'/',len(ids))
    X=np.concatenate(out).astype(np.float32)
    dump({'ids':list(ids),'X':X},cache); print('saved',cache,X.shape); return X

def style_motion(folder):
    paths=sorted(glob.glob(os.path.join(folder,'frame_*.jpg')))
    idx=np.linspace(0,len(paths)-1,12).round().astype(int)
    fs=[]; gs=[]
    for i in idx:
        im=cv2.imread(paths[i],cv2.IMREAD_COLOR)
        im=cv2.resize(im,(64,36),interpolation=cv2.INTER_AREA)
        rgb=cv2.cvtColor(im,cv2.COLOR_BGR2RGB).astype(np.float32)/255
        g=cv2.cvtColor(im,cv2.COLOR_BGR2GRAY).astype(np.float32)/255
        edge=cv2.Canny((g*255).astype(np.uint8),80,160).mean()/255
        h,w=g.shape; center=g[h//4:3*h//4,w//4:3*w//4]
        fs.append(np.r_[rgb.mean((0,1)),rgb.std((0,1)),g.mean(),g.std(),edge,center.mean()]); gs.append(g)
    A=np.stack(fs); D=np.stack([np.abs(a-b).mean() for a,b in zip(gs[:-1],gs[1:])])
    return np.r_[A.mean(0),A.std(0),A.min(0),A.max(0),D.mean(),D.std(),D.max()].astype(np.float32)

def style_features(ids, root):
    cache='/kaggle/working/'+Path(root).name+'_style.joblib'
    if os.path.exists(cache):
        obj=load(cache)
        if obj.get('ids')==list(ids): print('loaded',cache); return obj['X']
    X=[]
    for n,cid in enumerate(ids):
        X.append(style_motion(os.path.join(root,cid)))
        if (n+1)%250==0 or n==len(ids)-1: print('style features',n+1,'/',len(ids))
    X=np.stack(X); dump({'ids':list(ids),'X':X},cache); return X

Xtr=deep_features(train_ids,TRAIN,'train_r3d_features.joblib')
Xte=deep_features(test_ids,TEST,'test_r3d_features.joblib')
S_tr=style_features(train_ids,TRAIN); S_te=style_features(test_ids,TEST)
y=labels.label.astype(int).to_numpy(); groups=labels.group_id.to_numpy()
print('matrices:',Xtr.shape,Xte.shape,S_tr.shape,S_te.shape)

# Compare the pretrained temporal representation, style/motion model, and their OOF blend.
gkf=GroupKFold(n_splits=5); oof_deep=np.zeros(len(y)); oof_style=np.zeros(len(y));
for fold,(tr,va) in enumerate(gkf.split(Xtr,y,groups),1):
    deep=make_pipeline(StandardScaler(),LogisticRegression(C=0.15,max_iter=250,solver='lbfgs',random_state=SEED))
    style=HistGradientBoostingClassifier(max_iter=220,learning_rate=.05,max_leaf_nodes=31,l2_regularization=2.0,random_state=SEED+fold)
    deep.fit(Xtr[tr],y[tr]); style.fit(S_tr[tr],y[tr])
    oof_deep[va]=deep.predict_proba(Xtr[va])[:,1]; oof_style[va]=style.predict_proba(S_tr[va])[:,1]
    print('fold',fold,'deep',round(roc_auc_score(y[va],oof_deep[va]),5),'style',round(roc_auc_score(y[va],oof_style[va]),5))
print('OOF deep:',round(roc_auc_score(y,oof_deep),5),'style:',round(roc_auc_score(y,oof_style),5))
blends=[(a,roc_auc_score(y,a*oof_deep+(1-a)*oof_style)) for a in np.linspace(0,1,21)]
alpha,blend_score=max(blends,key=lambda t:t[1])
print('best OOF blend alpha(deep):',alpha,'ROC-AUC:',round(blend_score,5))

final_deep=make_pipeline(StandardScaler(),LogisticRegression(C=0.15,max_iter=300,solver='lbfgs',random_state=SEED))
final_style=HistGradientBoostingClassifier(max_iter=260,learning_rate=.05,max_leaf_nodes=31,l2_regularization=2.0,random_state=SEED)
final_deep.fit(Xtr,y); final_style.fit(S_tr,y)
pred=alpha*final_deep.predict_proba(Xte)[:,1]+(1-alpha)*final_style.predict_proba(S_te)[:,1]
pred=np.clip(pred,0.001,0.999)
out=sample[['clip_id']].copy(); out['label']=pred
out.to_csv('/kaggle/working/submission_improved.csv',index=False)
assert len(out)==len(sample) and list(out.columns)==['clip_id','label'] and out.label.between(0,1).all()
print(out.head()); print('READY improved file:',len(out),'rows; range',float(pred.min()),float(pred.max()))